# Day 8e — v3.1: adding graph-structural features

Attempt 3 (v3, `notebooks/08b_feature_fusion_classifier.ipynb`) fed the classifier a single flat category label — a real but marginal (~1%) improvement, with the ring features barely used by the model. Real production systems (per the Neo4j reference in `docs/hybrid-merge.md`) go further: **degree centrality and PageRank alongside the community/cluster ID**, not just the cluster ID alone.

This adds those two structural features — **no leakage risk, unlike the GNN-embedding attempt (v5)**: degree and PageRank are computed purely from graph structure, the same way for every node regardless of train/holdout membership, and never touch `isFraud`.

In [1]:
import sys, json
sys.path.append('..')

import numpy as np
import pandas as pd
import networkx as nx

from src.data import load_merged_train
from src.features import get_feature_lists
from src.split import apply_locked_split
from src.graph import load_graph
from src.hybrid import assign_cluster_categories
from src.model import build_xgb_pipeline
from src.cost import find_optimal_threshold, DEFAULT_REVIEW_COST
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

RANDOM_STATE = 42

## 1. Compute structural graph features for every transaction

In [2]:
train_full = load_merged_train()

identity_graph = load_graph('../data/processed/identity_graph.pkl')
behavioral_graph = load_graph('../data/processed/behavioral_graph.pkl')
combined_graph = load_graph('../data/processed/combined_graph.pkl')
with open('../data/processed/cluster_partition.json') as f:
    partition = {int(k): v for k, v in json.load(f).items()}

degree = dict(combined_graph.degree())
weighted_degree = dict(combined_graph.degree(weight='weight'))
pagerank = nx.pagerank(combined_graph, weight='weight')

ring_categories = assign_cluster_categories(train_full['TransactionID'], partition, identity_graph, behavioral_graph)
cluster_sizes = pd.Series(partition).value_counts()

train_full = train_full.set_index('TransactionID', drop=False)
train_full['ring_category'] = ring_categories
train_full['ring_cluster_size'] = train_full['TransactionID'].map(partition).map(cluster_sizes)
train_full['graph_degree'] = train_full['TransactionID'].map(degree)
train_full['graph_weighted_degree'] = train_full['TransactionID'].map(weighted_degree)
train_full['graph_pagerank'] = train_full['TransactionID'].map(pagerank)
train_full = train_full.reset_index(drop=True)

print('New feature summary:')
print(train_full[['graph_degree', 'graph_weighted_degree', 'graph_pagerank']].describe())

New feature summary:
        graph_degree  graph_weighted_degree  graph_pagerank
count  590540.000000          590540.000000    5.905400e+05
mean        3.637024               0.210756    1.693365e-06
std         7.292960               0.427500    8.938090e-07
min         0.000000               0.000000    8.013391e-07
25%         0.000000               0.000000    8.013391e-07
50%         1.000000               0.016486    1.641478e-06
75%         5.000000               0.222898    2.240700e-06
max       189.000000               8.600000    7.873896e-06


## 2. Feature lists: v3's features + the 3 new structural ones

In [3]:
numeric_features, categorical_features = get_feature_lists(train_full)
numeric_features = numeric_features + ['ring_cluster_size', 'graph_degree', 'graph_weighted_degree', 'graph_pagerank']
categorical_features = categorical_features + ['ring_category']

print(f'{len(numeric_features)} numeric features (incl. 4 graph-structural)')
print(f'{len(categorical_features)} categorical features (incl. ring_category)')

392 numeric features (incl. 4 graph-structural)
16 categorical features (incl. ring_category)


## 3. Train and evaluate on the same locked holdout

In [4]:
train_df, holdout_df = apply_locked_split(train_full)

X_train = train_df[numeric_features + categorical_features]
y_train = train_df['isFraud']
X_holdout = holdout_df[numeric_features + categorical_features]
y_holdout = holdout_df['isFraud']
amounts_holdout = holdout_df['TransactionAmt']

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model_v31 = build_xgb_pipeline(numeric_features, categorical_features, scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE)
model_v31.fit(X_train, y_train)
print('Fitted v3.1 classifier.')

Fitted v3.1 classifier.


In [5]:
y_proba_v31 = model_v31.predict_proba(X_holdout)[:, 1]

v31_threshold, v31_cost, _ = find_optimal_threshold(y_holdout, y_proba_v31, amounts_holdout, review_cost=DEFAULT_REVIEW_COST)

with open('../results/classifier_final_metrics.json') as f:
    locked_v2_metrics = json.load(f)
with open('../results/feature_fusion_metrics.json') as f:
    v3_metrics = json.load(f)
v2_cost = locked_v2_metrics['total_cost_rs']
v3_cost = v3_metrics['total_cost_rs']

print(f'v3.1 cost-optimal threshold: {v31_threshold:.2f}')
print(f'v3.1 cost: Rs {v31_cost:,.0f}')
print(f'v2 (locked, no graph features) cost: Rs {v2_cost:,.0f}')
print(f'v3 (flat category label only) cost: Rs {v3_cost:,.0f}')
print(f'v3.1 vs v2: Rs {v2_cost - v31_cost:,.0f} ({"v3.1 better" if v31_cost < v2_cost else "v2 better"})')
print(f'v3.1 vs v3: Rs {v3_cost - v31_cost:,.0f} ({"v3.1 better" if v31_cost < v3_cost else "v3 better"})')

v3.1 cost-optimal threshold: 0.84
v3.1 cost: Rs 330,948
v2 (locked, no graph features) cost: Rs 337,421
v3 (flat category label only) cost: Rs 333,872
v3.1 vs v2: Rs 6,473 (v3.1 better)
v3.1 vs v3: Rs 2,924 (v3.1 better)


In [6]:
y_pred_v31 = (y_proba_v31 >= v31_threshold).astype(int)
precision_v31 = precision_score(y_holdout, y_pred_v31)
recall_v31 = recall_score(y_holdout, y_pred_v31)
f1_v31 = f1_score(y_holdout, y_pred_v31)
pr_auc_v31 = average_precision_score(y_holdout, y_proba_v31)
cm_v31 = confusion_matrix(y_holdout, y_pred_v31)

print(f'Precision: {precision_v31:.4f} (v2: {locked_v2_metrics["precision"]:.4f}, v3: {v3_metrics["precision"]:.4f})')
print(f'Recall:    {recall_v31:.4f} (v2: {locked_v2_metrics["recall"]:.4f}, v3: {v3_metrics["recall"]:.4f})')
print(f'F1:        {f1_v31:.4f} (v2: {locked_v2_metrics["f1"]:.4f}, v3: {v3_metrics["f1"]:.4f})')
print(f'PR-AUC:    {pr_auc_v31:.4f} (v3: {v3_metrics["pr_auc"]:.4f})')
print('\nConfusion matrix:')
print(cm_v31)

Precision: 0.7971 (v2: 0.7582, v3: 0.8350)
Recall:    0.6417 (v2: 0.6593, v3: 0.6136)
F1:        0.7110 (v2: 0.7053, v3: 0.7074)
PR-AUC:    0.7547 (v3: 0.7566)

Confusion matrix:
[[113300    675]
 [  1481   2652]]


## 4. Did the structural features earn a bigger role than the flat label did?

In [7]:
preprocessor_v31 = model_v31.named_steps['prep']
classifier_v31 = model_v31.named_steps['clf']
feature_names_v31 = preprocessor_v31.get_feature_names_out()

importances = pd.Series(classifier_v31.feature_importances_, index=feature_names_v31).sort_values(ascending=False)

graph_feature_names = ['graph_degree', 'graph_weighted_degree', 'graph_pagerank', 'ring_cluster_size']
graph_importances = importances[importances.index.str.contains('graph_|ring_')]
print('Graph-derived feature importances and ranks:')
for name, value in graph_importances.items():
    rank = importances.index.get_loc(name) + 1
    print(f'  {name}: importance={value:.5f}, rank={rank} of {len(importances)}')

print('\nTop 15 features overall:')
print(importances.head(15))

Graph-derived feature importances and ranks:
  cat__ring_category_both: importance=0.00102, rank=186 of 563
  num__graph_weighted_degree: importance=0.00097, rank=200 of 563
  cat__ring_category_behavioral_only: importance=0.00094, rank=211 of 563
  num__ring_cluster_size: importance=0.00072, rank=259 of 563
  cat__ring_category_identity_only: importance=0.00067, rank=277 of 563
  num__graph_degree: importance=0.00066, rank=278 of 563
  num__graph_pagerank: importance=0.00063, rank=288 of 563
  cat__ring_category_isolated: importance=0.00000, rank=563 of 563

Top 15 features overall:
num__V258                       0.216789
num__V70                        0.101915
num__V201                       0.043833
num__V91                        0.043784
num__V294                       0.024101
cat__ProductCD_C                0.017599
cat__card6_debit                0.016395
num__V295                       0.014159
num__V102                       0.009194
num__C14                        0.009172

## Save results

In [8]:
v31_results = {
    'model': 'XGBoost v3.1 (ring_category + degree + weighted_degree + PageRank)',
    'threshold': v31_threshold,
    'precision': precision_v31,
    'recall': recall_v31,
    'f1': f1_v31,
    'pr_auc': pr_auc_v31,
    'confusion_matrix': cm_v31.tolist(),
    'total_cost_rs': v31_cost,
    'v2_locked_cost_rs': v2_cost,
    'v3_flat_label_cost_rs': v3_cost,
    'difference_vs_v2_rs': v2_cost - v31_cost,
    'difference_vs_v3_rs': v3_cost - v31_cost,
    'graph_feature_importances': graph_importances.to_dict(),
}

with open('../results/v3_1_structural_features_metrics.json', 'w') as f:
    json.dump(v31_results, f, indent=2, default=str)

print('Saved to results/v3_1_structural_features_metrics.json')

Saved to results/v3_1_structural_features_metrics.json


In [9]:
import joblib

joblib.dump(model_v31, '../models/classifier_v3_1.joblib')
print('Saved fitted model to models/classifier_v3_1.joblib')

Saved fitted model to models/classifier_v3_1.joblib


## Takeaway

_Fill in after running: whether v3.1 beats v2 and v3 on cost, whether degree/PageRank rank meaningfully higher than the flat category label did, and the final verdict on whether this is the right primary model._